# Supervised Model
- Collect images from dataset and transform into pixel array to train AI

### Load images from dataset in a vector

In [9]:
import os
import importlib
import CvrieLib.preprocessing as prep

importlib.reload(prep)
preprocess_image = prep.preprocess_image
flatten_image_array = prep.flatten_image_array

import numpy as np

X = []
Y = []
img_extension = '.jpg'

def load_fracture_dataset(directory, label_value, target_size=(224, 224)):
    files = [f for f in os.listdir(directory) if f.lower().endswith(img_extension)]
    n_files = len(files)
    print(f"Loading {n_files} files from {directory} with label {label_value}")
    data = np.zeros((n_files, target_size[0], target_size[1]), dtype=np.float32)
    labels = np.full(n_files, label_value, dtype=np.int8)

    for i, f in enumerate(files):
        full_path = os.path.join(directory, f)
        try:
            data[i] = preprocess_image(full_path, size=target_size)
        except Exception as e:
            print(f"Erreur dans le {f}: {e}")
    return data, labels


### Put label on data, and create X (the dataset ) and y the array of labels to keep track in next steps

In [10]:
X_fractured, y_fractured = load_fracture_dataset('Dataset/fractured/', label_value=1)
X_not_fractured, y_not_fractured = load_fracture_dataset('Dataset/not_fractured/', label_value=0)
# crer une pipeline avec scikit learn

X = np.concatenate([X_fractured, X_not_fractured], axis=0)
y = np.concatenate([y_fractured, y_not_fractured], axis=0)

print(f"Structure de X avant flatten : {X.shape}")

Loading 4840 files from Dataset/fractured/ with label 1
Loading 4623 files from Dataset/not_fractured/ with label 0
Structure de X avant flatten : (9463, 224, 224)


### Flatten the matrix in the matrix_array to obtain an array of array to comply model standard

In [11]:
X_flat = flatten_image_array(X)

print(f"Structure de X après flatten : {X_flat.shape}")

Structure de X après flatten : (9463, 50176)


#### Import scikit_learn

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report

### Split dataset in 80/20, 80% to train and 20% to test, shuffled equally with stratify

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=5, stratify=y)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (7570, 50176), y_train shape: (7570,)


### GridSearch on the dataset to find the best model to use

In [14]:
# param_grid_rf = {
#     'n_estimators': [100, 300, 500],
#     'max_depth': [10, 20, None],
#     'min_samples_split': [2, 5, 10],
#     'class_weight': ['balanced', None]
# }
#
# grid_rf = GridSearchCV(RandomForestClassifier(random_state=0), param_grid_rf, cv=5, scoring='f1', n_jobs=4)
#
# grid_rf.fit(X_train, y_train)
#
# print(f"Best RF Params: {grid_rf.best_params_}")
# print(f"Best RF F1-Score: {grid_rf.best_score_}")
#
# best_model = grid_rf.best_estimator_

In [15]:
# y_prediction = best_model.predict(X_test)
# print(confusion_matrix(y_test, y_prediction))
# print(classification_report(y_test, y_prediction))

### Train on best Model and gives scores

In [16]:
model = RandomForestClassifier(n_estimators=50, max_depth=5, n_jobs=-1, min_samples_split=2, min_samples_leaf=1)

# puis faire du gradient free routines (soit celui de scikit learn soit notre propre)
# pour choisir les hyperparameters du modele

model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)

print("Accuracy Score:", accuracy)
print(f"Fractured samples: {len(y_fractured)}")
print(f"Not fractured samples: {len(y_not_fractured)}")

# tester le model sur une image spécifique

Accuracy Score: 0.8230322239830956
Fractured samples: 4840
Not fractured samples: 4623
